<a href="https://colab.research.google.com/github/fantaxiah/vesicles-yolo/blob/color-analysis/JUNG_Copy_%EF%BC%92_of_vesicles_YOLO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Requirements

In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [2]:
!pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.4 MB/s eta 0:00:00


In [3]:
!pip install -q git+https://github.com/sunsmarterjie/yolov12.git roboflow supervision

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 109.1 MB/s eta 0:00:00


In [4]:
!pip install grad-cam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 60.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44286 sha256=faf3023a9c35e3f84c5dcd49922901d12e3453fa8f2429abdbb2472454e2ea49
  Stored in directory: /root/.cache/pip/wheels/fb/3b/09/2afc520f3d69bc26ae6bd87416759c820a3f7d05c1a077bbf6
Successfully built grad-cam


In [5]:
!pip install -q albumentations==1.4.7

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.7/155.7 kB 5.8 MB/s eta 0:00:00


In [6]:
#!pip uninstall -y torch torchvision torchaudio
#!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [7]:
#!pip uninstall -y torch torchvision torchaudio
#!pip cache purge

In [8]:
#!pip install torch==2.10.0 torchvision==0.17.2 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu128

In [9]:
import os
from google.colab import userdata
from google.colab import drive
import numpy as np
import torch
import torch.nn as nn
from torchvision.transforms import ToTensor
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import FasterRCNNBoxScoreTarget
import cv2
import os
import yaml
import random
import shutil
from ultralytics import YOLO
import supervision as sv
from roboflow import Roboflow
import matplotlib.pyplot as plt
from pathlib import Path
import albumentations as A


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/yolov12/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
FlashAttention is not available on this device. Using scaled_dot_product_attention instead.


# Dataset

In [10]:
# Load the API key from Colab userdata and set it as an environment variable
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

# Verify the API key is set
if "ROBOFLOW_API_KEY" not in os.environ or os.environ["ROBOFLOW_API_KEY"] is None:
    raise ValueError("ROBOFLOW_API_KEY environment variable is still not set after attempting to load from userdata.")
else:
  print("Roboflow API key loaded and environment variable set.")

Roboflow API key loaded and environment variable set.


In [11]:
rf = Roboflow(api_key=os.environ.get("ROBOFLOW_API_KEY"))
project = rf.workspace("myrines-workspace").project("vesicles-ipt-cy5-2")

loading Roboflow workspace...
loading Roboflow project...


In [12]:
version = 2
dataset = project.version(version).download("yolov8")  # or yolov5, yolov7, etc.


Extracting Dataset Version Zip to vesicles-ipt-cy5-2-2 in yolov8:: 100%|██████████| 57/57 [00:00<00:00, 970.98it/s]


In [13]:
#!rm -r vesicles-ipt-cy5-2-1/

In [14]:
!ls {dataset.location}

data.yaml  README.dataset.txt  README.roboflow.txt  train


In [15]:
!cat {dataset.location}/data.yaml

names:
- fat droplet
- non-spherical vesicle
- spherical vesicle
nc: 3
roboflow:
  license: CC BY 4.0
  project: vesicles-ipt-cy5-2
  url: https://universe.roboflow.com/myrines-workspace/vesicles-ipt-cy5-2/dataset/2
  version: 2
  workspace: myrines-workspace
test: ../test/images
train: ../train/images
val: ../valid/images


In [16]:
# # 'cmd' + '/' to comment and uncomment blocks of code
# base = "/content/vesicles-ipt-cy5-2-1"
# all_img_dir = os.path.join(base, "all_data", "images")
# all_lbl_dir = os.path.join(base, "all_data", "labels")

# os.makedirs(all_img_dir, exist_ok=True)
# os.makedirs(all_lbl_dir, exist_ok=True)

# for split in ["train_split", "val_split"]:
#     img_src = os.path.join(base, split, "images")
#     lbl_src = os.path.join(base, split, "labels")

#     for f in os.listdir(img_src):
#         shutil.copy2(os.path.join(img_src, f), os.path.join(all_img_dir, f))

#     for f in os.listdir(lbl_src):
#         shutil.copy2(os.path.join(lbl_src, f), os.path.join(all_lbl_dir, f))

# print("Merged images:", len(os.listdir(all_img_dir)))
# print("Merged labels:", len(os.listdir(all_lbl_dir)))

In [17]:
version_name = "vesicles-ipt-cy5-2-{}".format(version)
data_yaml_path = "/content/{}/data.yaml".format(version_name)

with open(data_yaml_path, "r") as f:
    data_yaml_content = yaml.safe_load(f)

# SETUP the yaml file to the correct file locations
data_yaml_content["train"] = "./train/images"
data_yaml_content["val"] = "./train/images"
if "test" in data_yaml_content:
    del data_yaml_content["test"]

with open(data_yaml_path, "w") as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)

!cat {data_yaml_path}

names:
- fat droplet
- non-spherical vesicle
- spherical vesicle
nc: 3
roboflow:
  license: CC BY 4.0
  project: vesicles-ipt-cy5-2
  url: https://universe.roboflow.com/myrines-workspace/vesicles-ipt-cy5-2/dataset/2
  version: 2
  workspace: myrines-workspace
train: ./train/images
val: ./train/images


# Image Augmentation

In [ ]:
# Root of the dataset (from Roboflow download)
root = Path(dataset.location)
print(root)
# We'll draw examples from train/images
train_img_dir = root / "train" / "images"
all_train_imgs = sorted(list(train_img_dir.glob("*.jpg")) + list(train_img_dir.glob("*.png")))

# Define an augmentation pipeline
augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.7),
  ])

def show_augmented_examples(n_examples=3):
    """Show original vs augmented vesicle images."""
    if len(all_train_imgs) == 0:
        print("No images found in train/images.")
        return

    else:
      print(f"Found {len(all_train_imgs)} training images")

    sample_paths = random.sample(all_train_imgs, min(n_examples, len(all_train_imgs)))

    plt.figure(figsize=(8, 4 * len(sample_paths)))

    for i, img_path in enumerate(sample_paths):
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        aug = augment(image=img_rgb)["image"]

        # original
        plt.subplot(len(sample_paths), 2, 2*i + 1)
        plt.imshow(img_rgb)
        plt.title(f"Original\n{img_path.name}")
        plt.axis("off")

        # augmented
        plt.subplot(len(sample_paths), 2, 2*i + 2)
        plt.imshow(aug)
        plt.title("Augmented")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

# Run this to see a few examples:
show_augmented_examples(4)


# YOLO Architecture

In [19]:
model = YOLO("yolov8n.pt")
# raw_model = model.model

100%|██████████| 6.25M/6.25M [00:00<00:00, 79.8MB/s]


In [20]:
results = model.val(
  data=data_yaml_path,
  epochs=300,
  # batch=8,
  # patience=20,
  # hsv_h=0.015,
  # hsv_s=0.7,
  # hsv_v=0.4,
  # translate=0.1,
  # scale=0.5,
  # fliplr=0.5,
  # mosaic=1.0,
  # mixup=0.5,
  # degrees=90,
  # shear=0.0,
  # perspective=0.0,
  # copy_paste=0.0
)

Ultralytics 8.3.63 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon 2.20GHz)
YOLOv8n summary (fused): 168 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs


100%|██████████| 755k/755k [00:00<00:00, 17.9MB/s]
val: Scanning /content/vesicles-ipt-cy5-2-2/train/labels... 27 images, 0 backgrounds, 0 corrupt: 100%|██████████| 27/27 [00:00<00:00, 48.23it/s]

val: WARNING ⚠️ /content/vesicles-ipt-cy5-2-2/train/images/4z_gL0-4-G77-2_frL0-1-G40-5_ALL_1669um-1-_jpg.rf.fafaa5f9394947a3cb3d5c4d217bdc93.jpg: 1 duplicate labels removed


val: New cache created: /content/vesicles-ipt-cy5-2-2/train/labels.cache


'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:19<00:00,  9.80s/it]


                   all         27      11705     0.0682    0.00106      0.036     0.0198
                person         24        723      0.087    0.00277     0.0482     0.0347
               bicycle         27       1518          0          0          0          0
                   car         27       9464      0.118   0.000423     0.0599     0.0246
Speed: 17.3ms preprocess, 463.7ms inference, 0.0ms loss, 11.9ms postprocess per image
Results saved to runs/detect/val


In [1]:
results = model.train(
    data=data_yaml_path,
    epochs=10,
    imgsz=1000
)

NameError: name 'model' is not defined

In [23]:
!ls runs/detect/train/weights
print(results.save_dir)
print(dataset.location)

results_file_path = results.save_dir

best.pt  last.pt
runs/detect/train5
/content/vesicles-ipt-cy5-2-2


## Image Predictions

In [ ]:
#class ImageScore(nn.Module):
#  def __init__(self, yolo_model):
#    super().__init__()
#    self.model = yolo_model

#  def forward(self, x):
#    preds = self.model(x)[0]
#    class_scores = preds[..., 4:]      #* preds[..., 5:].sigmoid()
#    max_conf = class_scores.max(dim=-1).values
#    image_score = max_conf.sum(dim=1)
#    return image_score.unsqueeze(-1)

In [24]:
ds = sv.DetectionDataset.from_yolo(
    images_directory_path=f"{dataset.location}/train/images",
    annotations_directory_path=f"{dataset.location}/train/labels",
    data_yaml_path=f"{dataset.location}/data.yaml"
)

KeyboardInterrupt: 

In [ ]:
best_file_path = f"{results_file_path}/weights/best.pt"
model = YOLO(best_file_path)

In [ ]:
image_path, image, target = ds[0]

results = model(image, conf=0.4, verbose=False)[0]
detections = sv.Detections.from_ultralytics(results)

image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

analysis_results = []

for box in detections.xyxy:
    result = analyze_detection(image_rgb, box, pixel_size_um=PIXEL_SIZE_UM)
    if result:
        analysis_results.append(result)

print(analysis_results)

print("Number of detections:", len(detections))
print("Confidences:", detections.confidence)

In [ ]:
from glob import glob


for i in range(len(ds)):
  image_path, image, target = ds[i]

  # preds
  results = model(image, conf=0.4, verbose=False)[0]  # LOWER threshold
  detections = sv.Detections.from_ultralytics(results).with_nms()

  print(f"Image {i} → Detections:", len(detections))  # DEBUG

  box_annotator = sv.BoxAnnotator(thickness=2)
  #label_annotator = sv.LabelAnnotator(text_scale=0.5)

  # annotate pred
  annotated_pred = image.copy()
  annotated_pred = box_annotator.annotate(scene=annotated_pred, detections=detections)
  #annotated_pred = label_annotator.annotate(scene=annotated_pred, detections=detections)

  # annotate ground truth
  annotated_gt = image.copy()
  annotated_gt = box_annotator.annotate(scene=annotated_gt, detections=target)
  #annotated_gt = label_annotator.annotate(scene=annotated_gt, detections=target)

  # plots
  plt.figure(figsize=(12, 6))

  plt.subplot(1, 2, 1)
  plt.imshow(cv2.cvtColor(annotated_gt, cv2.COLOR_BGR2RGB))
  plt.title("Ground Truth")
  plt.axis("off")

  plt.subplot(1, 2, 2)
  plt.imshow(cv2.cvtColor(annotated_pred, cv2.COLOR_BGR2RGB))
  plt.title("Predicted")
  plt.axis("off")

  plt.tight_layout()
  plt.show()

In [ ]:
image_path, image, target = ds[0]

results = model(image, conf=0.05)[0]
detections = sv.Detections.from_ultralytics(results)

print("Detections:", len(detections))

In [ ]:
false_positive_dir = "false_positives"
os.makedirs(false_positive_dir, exist_ok=True)

for i in range(len(ds)):
  image_path, image, target = ds[i]

  if target is not None and "boxes" in target and len(target["boxes"]) > 0:
    continue

  # run inference
  #results = model(image, verbose=False)[0]
  results = model(image, conf=0.1, verbose=False)[0]
  detections = sv.Detections.from_ultralytics(results).with_nms()

  print("Detections:", len(detections))

  # ff model pred
  if len(detections.xyxy) > 0:
    annotated_image = image.copy()
    annotated_image = sv.BoxAnnotator().annotate(scene=annotated_image, detections=detections)
    #annotated_image = sv.LabelAnnotator().annotate(scene=annotated_image, detections=detections)

    save_path = os.path.join(false_positive_dir, f"fp_{i}.jpg")
    cv2.imwrite(save_path, annotated_image[..., ::-1])


In [ ]:
folder = "false_positives"
image_files = sorted(os.listdir(folder))

for filename in image_files:
    img_path = os.path.join(folder, filename)
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.imshow(img)
    plt.title(filename)
    plt.axis('off')
    plt.show()

In [ ]:
!cat /content/vesicleimages-green-1/data.yaml


# Pixels to um

# Size Distribution

In [ ]:
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# microscope calibration – set correctly when you know it
PIXEL_SIZE_UM = 1.0  # µm per pixel

root = Path(dataset.location)
print("Dataset root:", root)


In [ ]:
# --- Cell A: size distribution from GT labels (train_split) ---

img_dir = root / "train_split" / "images"
lbl_dir = root / "train_split" / "labels"

img_files = sorted(list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")))
print(f"Found {len(img_files)} images in train_split/images")

records = []

for img_path in img_files:
    # find matching label file
    stem = img_path.stem
    lbl_path = lbl_dir / f"{stem}.txt"
    if not lbl_path.exists():
        continue

    img = cv2.imread(str(img_path))
    if img is None:
        continue

    h, w = img.shape[:2]

    with open(lbl_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue

            cls_id = int(parts[0])
            x_c_n, y_c_n, w_n, h_n = map(float, parts[1:5])  # normalized

            # convert to pixels
            w_px = w_n * w
            h_px = h_n * h
            diameter_px = (w_px + h_px) / 2.0
            radius_px = diameter_px / 2.0
            area_px2 = np.pi * (radius_px ** 2)

            # convert to microns
            diameter_um = diameter_px * PIXEL_SIZE_UM
            radius_um = radius_px * PIXEL_SIZE_UM
            area_um2 = area_px2 * (PIXEL_SIZE_UM ** 2)

            records.append({
                "source": "train_split_GT",
                "image": img_path.name,
                "class_id": cls_id,
                "width_px": w_px,
                "height_px": h_px,
                "diameter_px": diameter_px,
                "radius_px": radius_px,
                "area_px2": area_px2,
                "diameter_um": diameter_um,
                "radius_um": radius_um,
                "area_um2": area_um2,
            })

if not records:
    print("No GT boxes found.")
else:
    df_train_gt = pd.DataFrame(records)
    print(f"GT detections (train_split): {len(df_train_gt)}")
    display(df_train_gt.head())

    # histogram
    plt.figure(figsize=(6,4))
    plt.hist(df_train_gt["diameter_px"], bins=30)
    plt.xlabel("Vesicle diameter (pixels)")
    plt.ylabel("Count")
    plt.title("Size distribution – GT (train_split)")
    plt.tight_layout()
    plt.show()


In [ ]:
# --- Cell B: size distribution from YOLO predictions on augmented images ---

aug_dir = root / "train_aug_images"

if not aug_dir.exists():
    print("No augmented images folder found:", aug_dir)
else:
    aug_imgs = sorted(list(aug_dir.glob("*.jpg")) + list(aug_dir.glob("*.png")))
    print(f"Found {len(aug_imgs)} augmented images.")

    records_aug = []

    for img_path in aug_imgs:
        img = cv2.imread(str(img_path))
        if img is None:
            continue

        results = model(img, verbose=False)[0]
        if results.boxes is None or len(results.boxes) == 0:
            continue

        boxes_xyxy = results.boxes.xyxy.cpu().numpy()
        confidences = results.boxes.conf.cpu().numpy()
        class_ids = results.boxes.cls.cpu().numpy().astype(int)

        for box, conf, cls_id in zip(boxes_xyxy, confidences, class_ids):
            x1, y1, x2, y2 = box
            w_px = x2 - x1
            h_px = y2 - y1
            diameter_px = (w_px + h_px) / 2.0
            radius_px = diameter_px / 2.0
            area_px2 = np.pi * (radius_px ** 2)

            diameter_um = diameter_px * PIXEL_SIZE_UM
            radius_um = radius_px * PIXEL_SIZE_UM
            area_um2 = area_px2 * (PIXEL_SIZE_UM ** 2)

            records_aug.append({
                "source": "augmented_pred",
                "image": img_path.name,
                "class_id": cls_id,
                "confidence": conf,
                "width_px": w_px,
                "height_px": h_px,
                "diameter_px": diameter_px,
                "radius_px": radius_px,
                "area_px2": area_px2,
                "diameter_um": diameter_um,
                "radius_um": radius_um,
                "area_um2": area_um2,
            })

    if not records_aug:
        print("No detections on augmented images.")
    else:
        df_aug_pred = pd.DataFrame(records_aug)
        print(f"Detections (augmented images): {len(df_aug_pred)}")
        display(df_aug_pred.head())

        plt.figure(figsize=(6,4))
        plt.hist(df_aug_pred["diameter_px"], bins=30)
        plt.xlabel("Vesicle diameter (pixels)")
        plt.ylabel("Count")
        plt.title("Size distribution – YOLO predictions (augmented images)")
        plt.tight_layout()
        plt.show()


In [ ]:
# --- Cell C: size distribution from YOLO predictions on full dataset ---

image_dirs = [
    root / "train_split" / "images",
    root / "val_split" / "images",
    root / "test" / "images",
]

# include augmented images if exist
if (root / "train_aug_images").exists():
    image_dirs.append(root / "train_aug_images")

all_imgs = []
for d in image_dirs:
    if d.exists():
        imgs = sorted(list(d.glob("*.jpg")) + list(d.glob("*.png")))
        all_imgs.extend(imgs)
        print(f"{d}: {len(imgs)} images")

print(f"\nTotal images for full-dataset analysis: {len(all_imgs)}")

records_all = []

for img_path in all_imgs:
    img = cv2.imread(str(img_path))
    if img is None:
        continue

    results = model(img, verbose=False)[0]
    if results.boxes is None or len(results.boxes) == 0:
        continue

    boxes_xyxy = results.boxes.xyxy.cpu().numpy()
    confidences = results.boxes.conf.cpu().numpy()
    class_ids = results.boxes.cls.cpu().numpy().astype(int)

    for box, conf, cls_id in zip(boxes_xyxy, confidences, class_ids):
        x1, y1, x2, y2 = box
        w_px = x2 - x1
        h_px = y2 - y1
        diameter_px = (w_px + h_px) / 2.0
        radius_px = diameter_px / 2.0
        area_px2 = np.pi * (radius_px ** 2)

        diameter_um = diameter_px * PIXEL_SIZE_UM
        radius_um = radius_px * PIXEL_SIZE_UM
        area_um2 = area_px2 * (PIXEL_SIZE_UM ** 2)

        records_all.append({
            "source": "full_dataset_pred",
            "image": img_path.name,
            "class_id": cls_id,
            "confidence": conf,
            "width_px": w_px,
            "height_px": h_px,
            "diameter_px": diameter_px,
            "radius_px": radius_px,
            "area_px2": area_px2,
            "diameter_um": diameter_um,
            "radius_um": radius_um,
            "area_um2": area_um2,
        })

if not records_all:
    print("No detections on full dataset.")
else:
    df_all_pred = pd.DataFrame(records_all)
    print(f"Detections (full dataset): {len(df_all_pred)}")

    # Save combined CSV if you want
    out_csv = "vesicle_sizes_full_dataset.csv"
    df_all_pred.to_csv(out_csv, index=False)
    print("Saved:", out_csv)

    plt.figure(figsize=(6,4))
    plt.hist(df_all_pred["diameter_px"], bins=30)
    plt.xlabel("Vesicle diameter (pixels)")
    plt.ylabel("Count")
    plt.title("Size distribution – YOLO predictions (full dataset)")
    plt.tight_layout()
    plt.show()


# Color Analysis

In [ ]:
# =========================
# Vesicle analysis functions
# =========================

def get_membrane_mask(crop_rgb):
    gray = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    _, mask = cv2.threshold(
        blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    return mask


def fit_circle(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if len(contours) == 0:
        return None

    c = max(contours, key=cv2.contourArea)
    (x, y), r = cv2.minEnclosingCircle(c)

    return x, y, r


def classify_color(mean_r, mean_g, mean_b):
    eps = 1e-6
    rg = mean_r / (mean_g + eps)
    gr = mean_g / (mean_r + eps)

    if rg > 1.5:
        return "red"
    elif gr > 1.5:
        return "green"
    elif mean_r > 30 and mean_g > 30:
        return "mixed"
    else:
        return "dim"


def analyze_detection(image_rgb, box, pixel_size_um=1.0):
    x1, y1, x2, y2 = map(int, box)

    crop = image_rgb[y1:y2, x1:x2]
    if crop.size == 0:
        return None

    mask = get_membrane_mask(crop)

    # ---- SIZE (better than box)
    circle = fit_circle(mask)
    if circle is not None:
        _, _, r = circle
        diameter_px = 2 * r
    else:
        # fallback
        diameter_px = min(x2 - x1, y2 - y1)

    diameter_um = diameter_px * pixel_size_um

    # ---- COLOR (membrane only)
    pixels = crop[mask > 0]
    if len(pixels) == 0:
        return None

    mean_r = np.mean(pixels[:, 0])
    mean_g = np.mean(pixels[:, 1])
    mean_b = np.mean(pixels[:, 2])

    label = classify_color(mean_r, mean_g, mean_b)

    return {
        "diameter_px": float(diameter_px),
        "diameter_um": float(diameter_um),
        "mean_r": float(mean_r),
        "mean_g": float(mean_g),
        "mean_b": float(mean_b),
        "color": label
    }

In [ ]:
plt.hist(df["diameter_um"], bins=30)
plt.xlabel("Diameter (µm)")
plt.ylabel("Count")
plt.title("Vesicle Size Distribution")
plt.show()

# Imbedded GUI